# GPU 자원 극복 솔루션 🔥🔥🔥

1. 경량화된 LLM 활용
  - Mistral-7B, LLaMA 2 7B, Phi-2, TinyLlama, GPT-2/3 같은 작은 모델을 사용
  - QLoRA, GPTQ 등 저비트 양자화 적용 시 7B~13B 모델도 T4에서 사용 가능

2. 최적화된 파인튜닝 방식 활용
  - LoRA (Low-Rank Adaptation): 기존 LLM을 전체 미세조정하지 않고, 일부 가중치만 조정하여 VRAM 절약
  - QLoRA (4-bit LoRA): 양자화 + LoRA 조합으로 메모리 절약
  - PEFT (Parameter Efficient Fine-Tuning): 모델 전체가 아니라 일부 가중치만 학습하여 가벼운 미세조정 가능

3. LLM + RAG 조합
  - GPT-2, TinyLlama 등 작은 모델 + RAG (FAISS, ChromaDB) 조합

In [29]:
!pip install transformers peft accelerate bitsandbytes datasets huggingface_hub
!pip install pypdf tqdm sentencepiece torch
!pip install -U langchain-community

In [30]:
!pip install PyPDF2

In [31]:
!pip install faiss-cpu

In [32]:
!pip install sumy
!pip install lexrank
!pip install konlpy

In [33]:
pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [34]:
import pandas as pd
import numpy as np
import json
import os
import re
import warnings
warnings.filterwarnings('ignore')

import torch
from sentence_transformers import SentenceTransformer, util
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, T5ForConditionalGeneration, T5Tokenizer, T5EncoderModel
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from tqdm import tqdm
# from pypdf import PdfReader
from PyPDF2 import PdfReader
import faiss
# import fitz # metadata 페이지 번호만 제공
from rank_bm25 import BM25Okapi

from langchain.document_loaders import PyPDFLoader # 평균적으로 우수(한글 인코딩 처리, 속도, metadata)
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document
from langchain.retrievers import BM25Retriever
from langchain.schema.runnable import RunnablePassthrough

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer

from concurrent.futures import ThreadPoolExecutor, ThreadPoolExecutor

In [35]:
# from google.colab import drive

# drive.mount('/gdrive', force_remount=True)

In [36]:
# !unzip /gdrive/MyDrive/open.zip -d /gdrive/MyDrive/open

In [37]:
train_dir = '/kaggle/input/opendata/train.csv'
test_dir = '/kaggle/input/opendata/test.csv'
submission_dir = '/kaggle/input/opendata/sample_submission.csv'

train = pd.read_csv(train_dir, encoding='utf-8-sig')
test = pd.read_csv(test_dir)
submission = pd.read_csv(submission_dir, encoding='utf-8-sig')

**방지대책 생성에 필요한 컬럼**

- 공사종류
- 인적사고
- 물적사고
- 공종
- 사고객체
- 작업프로세스
- 장소
- 부위
- 사고원인

**불필요한 컬럼**

- 발생일시
- 사고인지시간
- 날씨
- 기온
- 습도
- 연면적
- 층 정보

In [38]:
train.head()

,ID,발생일시,사고인지 시간,날씨,기온,습도,공사종류,연면적,층 정보,인적사고,물적사고,공종,사고객체,작업프로세스,장소,부위,사고원인,재발방지대책 및 향후조치계획
0,TRAIN_00000,2023-12-31 오후 12:44,정규작업 -,맑음,1℃,30%,건축 / 건축물 / 근린생활시설,"4,892.77㎡","지상 14층, 지하 3층",떨어짐(5미터 이상 ~ 10미터 미만),없음,건축 > 철근콘크리트공사,건설자재 > 철근,설치작업,근린생활시설 / 내부,철근 / 고소,"고소작업 중 추락 위험이 있음에도 불구하고, 안전난간대, 안전고리 착용 등 안전장치...",고소작업 시 추락 위험이 있는 부위에 안전장비 설치.
1,TRAIN_00001,2023-12-30 오후 03:35,정규작업 -,맑음,10℃,90%,토목 / 터널 / 철도터널,-,-,끼임,없음,토목 > 터널공사,건설자재 > 볼트,운반작업,철도터널 / 내부,볼트 / 바닥,부주의,재발 방지 대책 마련과 안전교육 실시.
2,TRAIN_00002,2023-12-30 오후 02:30,정규작업 -,맑음,14℃,70%,건축 / 건축물 / 업무시설,"1,994.62㎡","지상 5층, 지하 0층",넘어짐(미끄러짐),없음,건축 > 철근콘크리트공사,기타 > 기타,형틀 및 목공,업무시설 / 내부,기타 / 바닥,3층 슬라브 작업시 이동중 미끄러짐,현장자재 정리와 안전관리 철저를 통한 재발 방지 대책 및 공문 발송을 통한 향후 조...
3,TRAIN_00003,2023-12-30 오후 12:00,휴일근무 -,흐림,12℃,55%,토목 / 하천 / 기타,-,-,기타,전도,토목 > 하천공사,부재 > 교각 기초,조립작업,기타 / 동산교 신축구간,교각 기초 / 바닥,"교각 기초철근 조립 중 강한 바람에 의해 기둥측 주철근이 균형을 잃고 전도되어, 하...","위험성 평가 및 교육을 통해 작업장 내 위험요인과 안전수칙을 근로자에게 전파하고, ..."
4,TRAIN_00004,2023-12-30 오전 10:00,정규작업 -,맑음,0℃,10%,건축 / 건축물 / 공동주택,"59,388.93㎡","지상 27층, 지하 3층",넘어짐(미끄러짐),없음,건축 > 해체 및 철거공사,기타 > 건설폐기물,해체작업,공동주택 / 내부,건설폐기물 / 바닥,근로자의 부주의,자재 정리 작업 시 세부 작업 방법에 대한 교육 실시와 작업 구간 이동 경로 점검 ...


In [39]:
train['공사종류(대분류)'] = train['공사종류'].str.split(' / ').str[0]
train['공사종류(중분류)'] = train['공사종류'].str.split(' / ').str[1]
train['공종(대분류)'] = train['공종'].str.split(' > ').str[0]
train['공종(중분류)'] = train['공종'].str.split(' > ').str[1]
train['사고객체(대분류)'] = train['사고객체'].str.split(' > ').str[0]
train['사고객체(중분류)'] = train['사고객체'].str.split(' > ').str[1]
train['장소(대분류)'] = train['장소'].str.split(' / ').str[0]
train['장소(중분류)'] = train['장소'].str.split(' / ').str[1]

test['공사종류(대분류)'] = test['공사종류'].str.split(' / ').str[0]
test['공사종류(중분류)'] = test['공사종류'].str.split(' / ').str[1]
test['공종(대분류)'] = test['공종'].str.split(' > ').str[0]
test['공종(중분류)'] = test['공종'].str.split(' > ').str[1]
test['사고객체(대분류)'] = test['사고객체'].str.split(' > ').str[0]
test['사고객체(중분류)'] = test['사고객체'].str.split(' > ').str[1]
test['장소(대분류)'] = test['장소'].str.split(' / ').str[0]
test['장소(중분류)'] = test['장소'].str.split(' / ').str[1]

In [40]:
needed_columns = ['공사종류', '인적사고', '물적사고', '공종', '사고객체', '작업프로세스', '장소', '부위', '사고원인', '재발방지대책 및 향후조치계획']

train['input_text'] = train.apply(lambda x: '\n'.join([f'{col}: {str(x[col])}' for col in needed_columns[:-1]]), axis=1)
train['output_text'] = train[needed_columns[-1]]
test['input_text'] = test.apply(lambda x: '\n'.join([f'{col}: {str(x[col])}' for col in needed_columns[:-1]]), axis=1)

In [41]:
def remove_unnecessary_chars(text):
  text = re.sub(r'\(.*?\)', '', text)
  text = re.sub(r'\[.*?\]', '', text)
  text = re.sub(r'<.*?>', '', text)
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'\s+', ' ', text)
  return text.strip()

In [42]:
def load_pdfs_with_pypdf2(pdf_dir):
    documents = []
    metadata = []

    for filename in os.listdir(pdf_dir):
        if filename.endswith(".pdf"):
            file_path = os.path.join(pdf_dir, filename)

            reader = PdfReader(file_path)
            for page_num, page in enumerate(reader.pages):
                if page_num < 2: # 첫 페이지 0 - 제목, 두 번째 페이지 1 - 목차 생략
                  continue
                text = page.extract_text()

                if text and text.strip():  # 빈 페이지 제외
                    processed_text = preprocess_pdf_text(text)
                    # processed_text = clean_pdf_text(processed_text)
                    processed_text = remove_unnecessary_chars(processed_text)
                    documents.append(processed_text)
                    metadata.append({"file": filename, "page": page_num + 1})

    return documents, metadata

In [43]:
def split_text(documents, metadata):
    splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
    chunk_texts = []
    chunk_metadata = []

    for text, meta in zip(documents, metadata):
        chunks = splitter.split_text(text)
        chunk_texts.extend(chunks)
        chunk_metadata.extend([meta] * len(chunks))  # 원본 문서 정보 유지

    return chunk_texts, chunk_metadata

In [44]:
def preprocess_pdf_text(pdf_text):
    """PDF 텍스트 전처리 (불필요한 특수문자 제거 등)"""
    # 특수 문자 제거, 불필요한 공백 처리, 줄바꿈 제거
    pdf_text = pdf_text.replace("\n", " ").replace("\r", "").strip()
    # 기타 불필요한 문자들 제거
    pdf_text = ''.join(e for e in pdf_text if e.isalnum() or e.isspace())
    return pdf_text

def preprocess_pdf_text(pdf_text):
    """PDF 텍스트 전처리: 영어, 숫자, 특수문자 제거"""
    pdf_text = pdf_text.replace("\n", " ").replace("\r", "").strip()

    # 영어, 숫자, 특수문자 제거 (한글, 공백만 남기기)
    pdf_text = re.sub(r'[^가-힣\s]', '', pdf_text)

    # 여러 개의 공백을 하나로 정리
    pdf_text = re.sub(r'\s+', ' ', pdf_text).strip()

    return pdf_text

# def clean_pdf_text(text):
#     # 'KOSHAGUIDE C' 또는 'KOSHAGuideC' 등의 변형을 제거 (대소문자 무시)
#     # cleaned_text = re.sub(r'(?i)KOSHAGUIDE\s*C\s*\d*부록\d*', '', text)
#     cleaned_text = re.sub(r'(?i)KOSHAGUIDE\s*C\s*\d*\s*부록?\d*', '', text)


#     # 여러 공백을 하나의 공백으로 정리
#     cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

#     return cleaned_text

In [57]:
# 키워드 중심 쿼리

def generate_query_text(row):

    # 공사종류: {row['공사종류(대분류)']} > {row['공사종류(중분류)']}
    # 공종: {row['공종(대분류)']} > {row['공종(중분류)']}
    # 장소: {row['장소(대분류)']} > {row['장소(중분류)']}
    # 사고객체: {row['사고객체(대분류)']} > {row['사고객체(중분류)']}
    # 인적사고: {row['인적사고']}
    # 물적사고: {row['물적사고']}
    # 작업 프로세스: {row['작업프로세스']}
    # 부위: {row['부위']}
    
    query = f"""
      인적사고: {row['인적사고']}
      작업 프로세스: {row['작업프로세스']}
      사고원인: {row['사고원인']}

      재발 방지 대책 및 향후 조치 계획을 제시해 주세요.
      """
    return query.strip()

In [47]:
PDF_DIR = '/kaggle/input/adddata/건설안전지침'
pdf_texts, pdf_metadata = load_pdfs_with_pypdf2(PDF_DIR)
pdf_texts, pdf_metadata = split_text(pdf_texts, pdf_metadata)

In [48]:
# # BM25 Retriever (키워드 기반 검색)
# def create_bm25_index(pdf_texts):
#   tokenized_texts = [text.split() for text in pdf_texts] # 문서 토큰화
#   bm25 = BM25Okapi(tokenized_texts)
#   return bm25

# def bm25_retrieve(query, bm25, pdf_texts, top_k=3):
#   tokenized_query = query.split() # 쿼리 토큰화
#   scores = bm25.get_scores(tokenized_query)
#   ranked_indices = np.argsort(scores)[::-1]
#   results = [(pdf_texts[i], scores[i]) for i in ranked_indices[:top_k]]
#   return results

# # def retrieve_top_pdfs(query, pdf_texts, pdf_metadata, top_k=5):
# #     query_emb = embedding_model.encode(query, convert_to_tensor=True)
# #     results = []

# #     for text, meta in zip(pdf_texts, pdf_metadata):
# #         pdf_emb = embedding_model.encode(text, convert_to_tensor=True)
# #         similarity = util.pytorch_cos_sim(query_emb, pdf_emb).squeeze().item()
# #         results.append((text, meta, similarity))

# #     results.sort(key=lambda x: x[2], reverse=True)
# #     return results[:top_k]

# # def retrieve_top_pdfs_with_faiss(query, top_k=3):
# #     query_emb = embedding_model.encode(query, convert_to_tensor=True).to("cuda")
# #     query_emb_np = query_emb.cpu().numpy().astype("float32")

# #     distances, indices = faiss_index.search(np.expand_dims(query_emb_np, axis=0), top_k)
# #     return [(pdf_texts[idx], pdf_metadata[idx]) for idx in indices[0] if idx != -1]

# # FAISS Retriever (의미 기반 검색)
# def create_faiss_index(pdf_embeddings):
#   faiss_index = faiss.IndexFlatL2(pdf_embeddings.shape[1])  # L2 거리 기준으로 FAISS 인덱스 생성
#   faiss_index.add(pdf_embeddings)  # 문서의 임베딩을 FAISS 인덱스에 추가
#   return faiss_index

# def faiss_retrieve(query, faiss_index, pdf_embeddings, pdf_texts, top_k=3):
#   query_emb = embedding_model.encode(query, convert_to_tensor=True).cpu().numpy().astype("float32")
#   distances, indices = faiss_index.search(np.expand_dims(query_emb, axis=0), top_k)
#   results = [(pdf_texts[idx], distances[0][i]) for i, idx in enumerate(indices[0])]
#   return results

# def normalize_scores(scores, method='max'):
#     if method == 'max':
#         max_score = max(scores) if scores else 1
#         return [score / max_score for score in scores]

#     elif method == 'zscore':
#         mean_score = np.mean(scores) if scores else 0
#         std_score = np.std(scores) if scores else 1
#         return [(score - mean_score) / std_score for score in scores]

#     return scores

# def hybrid_search(query, bm25, faiss_index, pdf_texts, pdf_embeddings, bm25_weight=0.5, faiss_weight=0.5, top_k=3):
#   # BM25 검색
#   bm25_results = bm25_retrieve(query, bm25, pdf_texts, top_k)

#   # FAISS 검색
#   faiss_results = faiss_retrieve(query, faiss_index, pdf_embeddings, pdf_texts, top_k)

#   # BM25 점수 정규화
#   bm25_scores = [score for _, score in bm25_results]
#   bm25_scores = normalize_scores(bm25_scores, method='max')
#   bm25_results = [(text, score) for (text, _), score in zip(bm25_results, bm25_scores)]

#   # FAISS 점수 정규화
#   faiss_scores = [score for _, score in faiss_results]
#   faiss_scores = normalize_scores(faiss_scores, method='max')
#   faiss_results = [(text, score) for (text, _), score in zip(faiss_results, faiss_scores)]

#   combined_results = bm25_results + faiss_results

#   combined_results = sorted(combined_results, key=lambda x: x[1], reverse=True)

#   return combined_results[:top_k]

# embedding_model = SentenceTransformer("jhgan/ko-sbert-nli")
# pdf_embeddings = embedding_model.encode(pdf_texts, convert_to_tensor=True).cpu().numpy().astype("float32")
# bm25 = create_bm25_index(pdf_texts)
# faiss_index = create_faiss_index(pdf_embeddings)

# top_k = 3
# score_threshold = 0.7
# retrieved_data = []

# for _, row in train.iterrows():
#   query_text = generate_query_text(row)

#   top_pdfs = hybrid_search(query_text, bm25, faiss_index, pdf_texts, pdf_embeddings, 0.5, 0.5, top_k)

#   filtered_pdfs = [(text, score) for text, score in top_pdfs if score >= score_threshold]

#   if filtered_pdfs:

#     for text, score in filtered_pdfs:
#       retrieved_data.append({"query": query_text, "retrieved_text": text, "summarized_text": summarize_text(text), "score": score})

# def combined_retrieved_text(df):
#   combined_texts = {}

#   for _, row in df.iterrows():
#     query = row['query']
#     summarized_text = row['summarized_text']

#     if query not in combined_texts:
#       combined_texts[query] = []

#     combined_texts[query].append(summarized_text)

#   return {q: ' '.join(t) for q, t in combined_texts.items()}

# combined_texts = combined_retrieved_text(pd.DataFrame(rag_data))

In [49]:
docs = [Document(page_content=text) for text in pdf_texts]
tokenized_corpus = [doc.page_content.split() for doc in docs]
bm25 = BM25Okapi(tokenized_corpus)

In [50]:
embedding_model = SentenceTransformer("jhgan/ko-sbert-nli")
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    # model_kwargs={'device': 'cpu'},
    # encode_kwargs={'normalize_embeddings': True}
)

doc_texts = [doc.page_content for doc in docs]
doc_vectors = embedding_model.encode(doc_texts, convert_to_tensor=True)

vector_store = FAISS.from_documents(docs, embeddings)
faiss_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 'score_threshold': 0.5, 일정 스코어 이상인 문서만 검색
# 'lambda_mult: 0.15, 유사도와 다양성 사이에서 적용될 수준 (관련성보다 다양성을 더 우선)
# 'search_type': 'mmr' 다양성을 고려한 MMR 검색
# 'fetch_k': 50, 후보 집합으로 선택되는 문서의 수

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [51]:
top_k = 3
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = top_k

def hybrid_search(query, bm25_retriever, faiss_retriever, bm25_weight=0.5, faiss_weight=0.5, top_k=3):
  ensemble_retriever = EnsembleRetriever(
      retrievers=[
          bm25_retriever,
          faiss_retriever
      ],
      weights=[bm25_weight, faiss_weight]
  )

  model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
  compressor = CrossEncoderReranker(model=model, top_n=top_k)

  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=ensemble_retriever
  )

  results = compression_retriever.invoke(query)[:top_k]

  return results

In [52]:
def summarize_text(text, max_sentences=10):
    """
    LexRank 알고리즘을 이용해 긴 텍스트를 요약하는 함수.
    - text: 요약할 원본 텍스트
    - max_sentences: 요약할 문장의 개수
    """
    parser = PlaintextParser.from_string(text, Tokenizer("korean"))
    # summarizer = LexRankSummarizer()
    summarizer = TextRankSummarizer()
    summary = summarizer(parser.document, max_sentences)

    return " ".join(str(sentence) for sentence in summary)

In [53]:
def cosine_similarity(a, b):
  dot_product = np.dot(a, b)
  norm_a = np.linalg.norm(a)
  norm_b = np.linalg.norm(b)
  return dot_product / (norm_a * norm_b) if norm_a != 0 and norm_b != 0 else 0

def compute_similarity_score(query, retrieved_text, model, tokenizer):
  query_embedding = model.encode([query])
  retrieved_embedding = model.encode([retrieved_text])

  score = cosine_similarity(query_embedding[0], retrieved_embedding[0])
  return score

In [54]:
# top_k = 3
# retrieved_data = []
# similarity_threshold = 0.6
# model = SentenceTransformer('jhgan/ko-sbert-nli')

# for _, row in train.iterrows():
#   query_text = generate_query_text(row)

#   top_pdfs = hybrid_search(query_text, bm25_retriever, faiss_retriever, 0.5, 0.5, top_k)

#   retrieved_text = ' '.join([doc.page_content for doc in top_pdfs])

#   similarity_score = compute_similarity_score(query_text, retrieved_text, model, model.tokenizer)
#   if similarity_score < similarity_threshold:

#   retrieved_data.append(retrieved_text)

In [ ]:
# MultiProcessing

top_k = 3
retrieved_data = []
similarity_threshold = 0.6
model = SentenceTransformer('jhgan/ko-sbert-nli')

def retrieve_process(row):
  query_text = generate_query_text(row)
  top_pdfs = hybrid_search(query_text, bm25_retriever, faiss_retriever, 0.5, 0.5, top_k)

  retrieved_text = ' '.join([doc.page_content for doc in top_pdfs])

  similarity_score = compute_similarity_score(row['재발방지대책 및 향후조치계획'], retrieved_text, model, model.tokenizer)

  if similarity_score >= similarity_threshold:
    combined_text = row['재발방지대책 및 향후조치계획'] + ' ' + retrieved_text
  else:
    combined_text = row['재발방지대책 및 향후조치계획']

  return combined_text

# multiprocessing.set_start_method('spawn', force=True)
# multiprocessing.get_start_method()

# ProcessPoolExecutor(max_workers=4), ThreadPoolExecutor(max_workers=4)

if __name__ == '__main__':
  with ThreadPoolExecutor(max_workers=4) as executor:
    retrieved_data = list(executor.map(retrieve_process, [row[1] for row in train.iterrows()]))

In [ ]:
def batch_compute_similarity(rows, texts, model):
    row_embeddings = model.encode(rows, convert_to_tensor=True)
    text_embeddings = model.encode(texts, convert_to_tensor=True)
    similarity_scores = (row_embeddings @ text_embeddings.T).diag()  # 코사인 유사도 계산
    return similarity_scores.cpu().tolist()

In [ ]:
top_k = 3
retrieved_data = []
similarity_threshold = 0.6
model = SentenceTransformer('jhgan/ko-sbert-nli')

def batch_retrieve_process(rows):
    queries = [generate_query_text(row) for _, row in rows.iterrows()]
    top_pdfs_batch = [hybrid_search(query, bm25_retriever, faiss_retriever, 0.5, 0.5, top_k) for query in queries]

    retrieved_texts = [' '.join([doc.page_content for doc in top_pdfs]) for top_pdfs in top_pdfs_batch]

    similarity_scores = batch_compute_similarity(
        [row['재발방지대책 및 향후조치계획'] for row in rows.iterrows()],
        retrieved_texts,
        model
    )

    results = [
        row['재발방지대책 및 향후조치계획'] + ' ' + retrieved_text if score >= similarity_threshold else row['재발방지대책 및 향후조치계획']
        for row, retrieved_text, score in zip(rows, retrieved_texts, similarity_scores)
    ]
    return results

if __name__ == '__main__':
    batch_size = 100
    row_batches = [train.iloc[i:i+batch_size] for i in range(0, len(train), batch_size)]

    with ThreadPoolExecutor(max_workers=8) as executor:
        retrieved_data = list(tqdm(executor.map(batch_retrieve_process, row_batches), total=len(row_batches)))

  0%|          | 0/235 [00:00<?, ?it/s]

In [ ]:
retrieved_data

In [61]:
# MultiProcessing

top_k = 2

def retrieve_process(row):
  query_text = generate_query_text(row)
  top_pdfs = hybrid_search(query_text, bm25_retriever, faiss_retriever, 0.5, 0.5, top_k)

  retrieved_text = ' '.join([doc.page_content for doc in top_pdfs])

  # similarity_score = compute_similarity_score(query_text, retrieved_text, model, model.tokenizer)

  # if similarity_score < similarity_threshold:
  #   retrived_text = row['재발방지대책 및 향후조치계획']

  return retrieved_text

# ProcessPoolExecutor(max_workers=4), ThreadPoolExecutor(max_workers=4)

with ThreadPoolExecutor(max_workers=8) as executor:
  retrieved_data = list(tqdm(executor.map(retrieve_process, [row for _, row in train.iterrows()]), total=len(train)))

  0%|          | 0/964 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 1/964 [00:07<1:59:05,  7.42s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (1,370 outputs left out)

100%|█████████▉| 961/964 [30:12<00:05,  1.92s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 964/964 [30:15<00:00,  1.88s/it]


In [64]:
import joblib

joblib.dump(retrieved_data, 'retrieved_data.pkl')

['retrieved_data.pkl']

In [ ]:
# joblib.load('retrieved_data.csv')

In [ ]:
train['재발방지대책 및 향후조치계획'].value_counts()

,count
재발방지대책 및 향후조치계획,
안전교육 실시를 통한 재발 방지 대책 및 향후 조치 계획.,76
작업자 안전교육 실시를 통한 재발 방지 대책 및 향후 조치 계획.,53
안전관리교육철저에 대한 재발 방지 대책과 향후 조치 계획.,39
안전교육 철저에 대한 재발 방지 대책과 향후 조치 계획.,36
동일한 사고가 발생하지 않도록 안전교육 실시.,32
...,...
작업자 교육 및 주변 정리 철저에 따른 향후 조치 계획.,1
작업자 상태 지속적 파악과 공사현장 사고 재발 방지를 위한 현장 관리 철저.,1
해당 사고사례 전파교육 및 작업전 교육 철저 실시,1


In [ ]:
model_name = "google/mt5-small"  # t5-small # t5-base # lassl/ko-t5-small
# tokenizer = T5Tokenizer.from_pretrained(model_name)
# model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = MT5Tokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name)

def generate_prevention_plan(query_text, retrieved_texts, max_length=5000):
    """한국어 MT5 모델을 활용하여 대책 생성"""
    input_text = f"대책 생성: {retrieved_texts}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
    output_ids = model.generate(**inputs, max_length=max_length, no_repeat_ngram_size=2, num_beams=5, temperature=0.9, top_p=0.85)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


In [ ]:
# final_plans = {query_text: generate_prevention_plan(query_text, retrieved_texts) for query_text, retrieved_texts in combined_texts.items()}

스코어 threshold 이상이면 결합

상위 문서 내용을 뽑아서 chatgpt api가 ex) "대책을 생성해줘" 프롬프트에 대한 대책을 생성?

jaccard_similarity - 아시다시피 생성된 답변을 정량적으로 평가하는 지표가 아직까지 명확하지 않다고 생각했기 때문에

			생성된 답변의 정성적 지표를 정량적 지표로 어떻게 간격을 줄이느냐가 모델 성능 개선에 큰 역할을 한 것 같습니다.

			확률적 샘플링을 통해 생성된 답변 후보군 중에 저희만의 정량적 지표로 가장 퀄리티 높은 답변 하나를 선택한 전략으로 성과를 만들어냈습니다.

저는 데이터셋이 일정한 규칙을 가지고 만들어졌다기 보단
약간 사고 원인을 바탕으로 포괄적인 대책이 레이블링 되어 있다고 생각했습니다

In [ ]:
# train['output_text'].value_counts()[train['output_text'].value_counts() > 1]

In [ ]:
# train['output_text'].value_counts()[train['output_text'].value_counts() < 2]

# Retrieval

In [76]:
needed_columns = ['인적사고', '사고원인']

train["combined_text"] = train[needed_columns].fillna("").agg(" ".join, axis=1)
test["combined_text"] = test[needed_columns].fillna("").agg(" ".join, axis=1)

sbert_model = SentenceTransformer('jhgan/ko-sbert-nli')

train_embeddings = sbert_model.encode(train["combined_text"].tolist(), convert_to_tensor=True).cpu().numpy()

dimension = train_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(train_embeddings)

In [ ]:
def find_similar_case(test_text, top_k=3):
    test_embedding = sbert_model.encode([test_text], convert_to_tensor=True).cpu().numpy()
    distances, indices = index.search(test_embedding, top_k)

    return train.iloc[indices[0]]["재발방지대책 및 향후조치계획"].tolist()

test["retrieved_cases"] = test["combined_text"].apply(find_similar_case)

In [ ]:
test['retrieved_cases'].head()

In [ ]:
test["retrieved_cases"][0]

In [77]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import MinMaxScaler

needed_columns = ['인적사고', '사고원인']

# 데이터 결합
train["combined_text"] = train[needed_columns].fillna("").agg(" ".join, axis=1)
test["combined_text"] = test[needed_columns].fillna("").agg(" ".join, axis=1)

# SBERT 모델 로드
sbert_model = SentenceTransformer('jhgan/ko-sbert-nli')

# Train 데이터 임베딩
train_embeddings = sbert_model.encode(train["combined_text"].tolist(), convert_to_tensor=True).cpu().numpy()

# FAISS 인덱스 설정
dimension = train_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(train_embeddings)

# BM25 설정
tokenized_corpus = [text.split() for text in train["combined_text"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

# FAISS와 BM25 검색을 위한 함수
def find_similar_case_with_bm25_faiss(test_text, top_k=3, bm25_weight=0.5, faiss_weight=0.5):
    # BM25 검색
    tokenized_query = test_text.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_top_indices = np.argsort(bm25_scores)[::-1]

    # FAISS 검색
    test_embedding = sbert_model.encode([test_text], convert_to_tensor=True).cpu().numpy()
    distances, indices = index.search(test_embedding, len(train))

    faiss_scores = 1 / (1 + distances.flatten())  # L2 - 거리가 낮을수록 높은 유사도 따라서 반비례 수행

    # BM25와 FAISS 스코어 정규화 (Min-Max Scaling)
    scaler = MinMaxScaler()
    bm25_scores_normalized = scaler.fit_transform(bm25_scores.reshape(-1, 1)).flatten()
    faiss_scores_normalized = scaler.fit_transform(faiss_scores.reshape(-1, 1)).flatten()

    # 가중치 적용하여 결합
    combined_scores = bm25_weight * bm25_scores_normalized + faiss_weight * faiss_scores_normalized

    # 최종 결합된 스코어로 상위 k개 문서 선택
    top_indices = np.argsort(combined_scores)[::-1][:top_k]

    faiss_idx = indices[0][0]  # FAISS에서의 상위 인덱스 선택
    selected_cases = train.iloc[faiss_idx]["재발방지대책 및 향후조치계획"]

    # selected_cases = []
    # for idx in top_indices:
    #     # BM25와 FAISS에서 동일한 문서를 가리키기 위해 인덱스를 조정
    #     # indices[0][idx]로 FAISS 인덱스를 얻어 올바른 train 인덱스를 찾음
    #     faiss_idx = indices[0][idx]  # FAISS에서의 상위 인덱스 선택
    #     selected_cases.append(train.iloc[faiss_idx]["재발방지대책 및 향후조치계획"])

    return selected_cases

# test 데이터에서 검색 수행
test["retrieved_cases"] = test["combined_text"].apply(find_similar_case_with_bm25_faiss)

print(test[["combined_text", "retrieved_cases"]])

Batches:   0%|          | 0/732 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (956 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

                                         combined_text  \
0    부딪힘 펌프카 아웃트리거 바닥 고임목을 3단으로 보강 했음에도, 지반 침하(아웃트리...   
1             절단, 베임 작업자의 불안전한 행동(숫돌 측면 사용) 및 보안면 미 착용   
2    떨어짐(2미터 미만) 작업자가 작업을 위해 이동 중 전방을 주시하지 않아 발을 헛디...   
3                    넘어짐(물체에 걸림) 작업 발판 위 벽돌 잔재를 밟고 넘어짐   
4         떨어짐(2미터 미만) 점심식사를 위한 이동시 작업자 부주의로 인한 추락사고 발생   
..                                                 ...   
959                         부딪힘 약한 석재가 자연 파단하여 늑골 골절 됨   
960                   넘어짐(기타) 근로자가 작업 중 이동가능한 통로 확보 미흡   
961   끼임 작업자가 믹서트럭 슈트아래로 불안전하게 이동하면서 콘크리트 통사이에 손가락이 끼임   
962             기타 깨진타일을 손으로 주워 마대자루에 넣을려고 하던 중 파편이 박힘   
963  절단, 베임 2024년 11월 6일 9시경 재해자가 혼자 세대 천정 작업 중 각도 ...   

                                       retrieved_cases  
0    작업자 안전교육 철저 및 시스템동바리 안전망 추가 설치와 지속적인 작업자 안전교육 ...  
1    근로자 및 관리감독자 교육 강화와 당일 16:30 병원 이송, 8.5 정밀검사, 8...  
2    협력업체 조출작업 관리 철저, 작업장 이동통로 설치 및 관리, 조도 확보, 위험요인...  
3               주기적 안전교육 실시와 불안전한 자세 및 행동에 대한 관리감독 철저.  
4    사고 방지를 위한 안전교

In [78]:
embedding_model_name = "jhgan/ko-sbert-sts"
embedding = SentenceTransformer(embedding_model_name)

pred_embeddings = embedding.encode(test["retrieved_cases"])

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

In [79]:
test['retrieved_data'] = retrieved_data

In [65]:
embedding_model_name = "jhgan/ko-sbert-sts"
embedding = SentenceTransformer(embedding_model_name)

pred_embeddings = embedding.encode(retrieved_data)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

In [66]:
pred_embeddings

array([[ 0.15478995,  0.429385  , -0.14756916, ...,  0.18559952,
        -0.77788967, -0.19451986],
       [ 0.15478995,  0.429385  , -0.14756916, ...,  0.18559952,
        -0.77788967, -0.19451986],
       [ 0.35897744,  0.70405006, -0.44479886, ..., -0.25510076,
        -0.5390669 ,  0.10552892],
       ...,
       [ 0.15478995,  0.429385  , -0.14756916, ...,  0.18559952,
        -0.77788967, -0.19451986],
       [ 0.69158036,  0.35514358, -0.48689982, ..., -0.05876052,
        -0.66049826,  0.02518005],
       [ 0.5095717 ,  0.8136462 , -0.3593986 , ...,  0.12732586,
        -1.0454504 ,  0.48097855]], dtype=float32)

In [67]:
submission.iloc[:,1] = retrieved_data # test["retrieved_cases"]
submission.iloc[:,2:] = pred_embeddings
submission.head()

,ID,재발방지대책 및 향후조치계획,vec_0,vec_1,vec_2,vec_3,vec_4,vec_5,vec_6,vec_7,...,vec_758,vec_759,vec_760,vec_761,vec_762,vec_763,vec_764,vec_765,vec_766,vec_767
0,TEST_000,협소한장소에서작업이수행됨에따라떨어짐끼임부딪힘등의재해가 발생할수있으므로작업반경등을고려...,0.154790,0.429385,-0.147569,1.330870,-0.443849,0.215196,0.671244,-0.090248,...,0.136858,1.692628,0.207999,0.829833,0.016680,-0.482606,0.290462,0.185600,-0.777890,-0.194520
1,TEST_001,협소한장소에서작업이수행됨에따라떨어짐끼임부딪힘등의재해가 발생할수있으므로작업반경등을고려...,0.154790,0.429385,-0.147569,1.330870,-0.443849,0.215196,0.671244,-0.090248,...,0.136858,1.692628,0.207999,0.829833,0.016680,-0.482606,0.290462,0.185600,-0.777890,-0.194520
2,TEST_002,추진관내에서작업하는근로자의안전보건에위해되지않도록작업전에 관내의산소농도유해가스먼지등의...,0.358977,0.704050,-0.444799,0.710849,-0.622906,-0.210348,0.819985,-0.121243,...,-0.197581,1.293195,0.219359,0.516017,-0.143075,-0.545459,-0.023751,-0.255101,-0.539067,0.105529
3,TEST_003,협소한장소에서작업이수행됨에따라떨어짐끼임부딪힘등의재해가 발생할수있으므로작업반경등을고려...,0.154790,0.429385,-0.147569,1.330870,-0.443849,0.215196,0.671244,-0.090248,...,0.136858,1.692628,0.207999,0.829833,0.016680,-0.482606,0.290462,0.185600,-0.777890,-0.194520
4,TEST_004,추진관내에서작업하는근로자의안전보건에위해되지않도록작업전에 관내의산소농도유해가스먼지등의...,0.358977,0.704050,-0.444799,0.710849,-0.622906,-0.210348,0.819985,-0.121243,...,-0.197581,1.293195,0.219359,0.516017,-0.143075,-0.545459,-0.023751,-0.255101,-0.539067,0.105529


In [69]:
submission.to_csv('./rag_submission.csv', index=False, encoding='utf-8-sig')

In [70]:
from transformers import pipeline

# model_name = "beomi/llama-3-ko-4b" # "NCSOFT/Llama-VARCO-8B-Instruct" # "MLP-KTLim/llama-3-Korean-Bllossom-8B"
# text_generator = pipeline("text-generation", model=model_name, tokenizer=model_name, device=0)

model_name = "NCSOFT/Llama-VARCO-8B-Instruct"

# 모델 로딩 시 4bit 사용
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # float16 사용으로 RAM 절약
    device_map="auto",          # GPU 자동 할당
    quantization_config={"load_in_4bit": True}  # 4bit 로드
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/51.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# row['retrieved_cases'], retrieved_data

In [80]:
test.head()

,ID,발생일시,사고인지 시간,날씨,기온,습도,공사종류,연면적,층 정보,인적사고,...,공종(대분류),공종(중분류),사고객체(대분류),사고객체(중분류),장소(대분류),장소(중분류),input_text,combined_text,retrieved_cases,retrieved_data
0,TEST_000,2024-06-03 오전 09:39,정규작업 -,맑음,27℃,53%,건축 / 건축물 / 교정 및 군사시설,"1,990.32㎡","지상 1층, 지하 0층",부딪힘,...,건축,철근콘크리트공사,건설기계,콘크리트펌프,교정 및 군사시설,외부,공사종류: 건축 / 건축물 / 교정 및 군사시설\n인적사고: 부딪힘\n물적사고: 전...,"부딪힘 펌프카 아웃트리거 바닥 고임목을 3단으로 보강 했음에도, 지반 침하(아웃트리...",작업자 안전교육 철저 및 시스템동바리 안전망 추가 설치와 지속적인 작업자 안전교육 ...,협소한장소에서작업이수행됨에따라떨어짐끼임부딪힘등의재해가 발생할수있으므로작업반경등을고려...
1,TEST_001,2024-02-15 오전 09:00,정규작업 -,맑음,5℃,71%,건축 / 건축물 / 운수시설,"349,895㎡","지상 5층, 지하 2층","절단, 베임",...,건축,수장공사,건설공구,공구류,운수시설,내부,"공사종류: 건축 / 건축물 / 운수시설\n인적사고: 절단, 베임\n물적사고: 없음\...","절단, 베임 작업자의 불안전한 행동(숫돌 측면 사용) 및 보안면 미 착용","근로자 및 관리감독자 교육 강화와 당일 16:30 병원 이송, 8.5 정밀검사, 8...",협소한장소에서작업이수행됨에따라떨어짐끼임부딪힘등의재해가 발생할수있으므로작업반경등을고려...
2,TEST_002,2024-02-01 오전 09:30,작업 전 -,흐림,5℃,58%,건축 / 건축물 / 공동주택,"171,198㎡","지상 27층, 지하 2층",떨어짐(2미터 미만),...,건축,미장공사,기타,기타,공동주택,내부,공사종류: 건축 / 건축물 / 공동주택\n인적사고: 떨어짐(2미터 미만)\n물적사고...,떨어짐(2미터 미만) 작업자가 작업을 위해 이동 중 전방을 주시하지 않아 발을 헛디...,"협력업체 조출작업 관리 철저, 작업장 이동통로 설치 및 관리, 조도 확보, 위험요인...",추진관내에서작업하는근로자의안전보건에위해되지않도록작업전에 관내의산소농도유해가스먼지등의...
3,TEST_003,2024-02-06 오후 05:00,정규작업 -,맑음,1℃,87%,건축 / 건축물 / 공동주택,"123,540.98㎡","지상 2층, 지하 35층",넘어짐(물체에 걸림),...,건축,조적공사,건설자재,자재,공동주택,102동 801호 세대 내부,공사종류: 건축 / 건축물 / 공동주택\n인적사고: 넘어짐(물체에 걸림)\n물적사고...,넘어짐(물체에 걸림) 작업 발판 위 벽돌 잔재를 밟고 넘어짐,주기적 안전교육 실시와 불안전한 자세 및 행동에 대한 관리감독 철저.,협소한장소에서작업이수행됨에따라떨어짐끼임부딪힘등의재해가 발생할수있으므로작업반경등을고려...
4,TEST_004,2024-06-03 오전 11:35,기타 -,흐림,22℃,60%,토목 / 교량 / 도로교량,-,-,떨어짐(2미터 미만),...,토목,교량공사,기타,기타,도로교량,외부,공사종류: 토목 / 교량 / 도로교량\n인적사고: 떨어짐(2미터 미만)\n물적사고:...,떨어짐(2미터 미만) 점심식사를 위한 이동시 작업자 부주의로 인한 추락사고 발생,"사고 방지를 위한 안전교육 실시, 안전보호구 착용 철저 감독, 위험 작업 시 2인 ...",추진관내에서작업하는근로자의안전보건에위해되지않도록작업전에 관내의산소농도유해가스먼지등의...


Llama 모델은 [INST]를 넣어주는게 핵심 좋음.

In [ ]:
def generate_countermeasure(row):
    # 유사 대책 3개 가져오기
    similar_cases = " ".join(row.get('retrieved_cases', []))  # 리스트일 경우 대비
    pdf_retrieved_cases = row.get('retrieved_data', '')

    context_data = "\n\n".join(filter(None, [similar_cases, pdf_retrieved_cases]))

    # 텍스트 생성 프롬프트 설정
    prompt = f"""
    [INST]
    ### 역할:
    당신은 건설 현장의 안전 전문가입니다.  
    주어진 사고 사례와 유사한 사고 사례를 참고하여, 최적의 재발 방지 대책을 제시하세요.  
    **- 간략하지만 실질적인 조치를 포함하세요.**  
    **- 서론이나 배경 설명 없이 바로 해결책을 제시하세요.**  
    **- "제안합니다" 등의 문구는 사용하지 말고, 직접적인 조치 사항을 기술하세요.** 
    
    ### 사고 정보:
    - 인적사고: {row['인적사고']}
    - 작업프로세스: {row['작업프로세스']}
    - 사고원인: {row['사고원인']}

    ### 참고할 유사 사고 사례에 대한 대책:
    {context_data if context_data.strip() else "없음"}

    [/INST]
    """

    # 텍스트 생성
    # generated_text = text_generator(prompt, max_length=150, do_sample=True, top_p=0.9, truncation=True)[0]["generated_text"]
    generated_text = text_generator(prompt, max_new_tokens=128, do_sample=True, top_p=0.9)[0]["generated_text"]

    return generated_text.strip()

test["generated_countermeasure"] = test.apply(generate_countermeasure, axis=1)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
from transformers import pipeline

def generate_countermeasure_batch(rows):
    prompts = []

    for _, row in rows.iterrows():
        similar_cases = " ".join(row.get('retrieved_cases', []))
        pdf_retrieved_cases = row.get('retrieved_data', '')

        context_data = "\n\n".join(filter(None, [similar_cases, pdf_retrieved_cases]))

        prompt = f"""
        [INST]
        ### 역할:
        당신은 건설 현장의 안전 전문가입니다.  
        주어진 사고 사례와 유사한 사고 사례를 참고하여, 최적의 재발 방지 대책을 제시하세요.  
        **- 간략하지만 실질적인 조치를 포함하세요.**  
        **- 서론이나 배경 설명 없이 바로 해결책을 제시하세요.**  
        **- "제안합니다" 등의 문구는 사용하지 말고, 직접적인 조치 사항을 기술하세요.** 
        
        ### 사고 정보:
        - 인적사고: {row['인적사고']}
        - 작업프로세스: {row['작업프로세스']}
        - 사고원인: {row['사고원인']}

        ### 참고할 유사 사고 사례에 대한 대책:
        {context_data if context_data.strip() else "없음"}

        [/INST]
        """
        prompts.append(prompt.strip())

    # 🚀 **배치 생성 실행**
    generated_texts = text_generator(prompts, max_new_tokens=128, do_sample=True, top_p=0.9)

    # Llama 모델의 출력에서 실제 생성된 텍스트 추출
    return [gen[0]["generated_text"].strip() for gen in generated_texts]

# 🚀 **apply 대신 batch 처리**
batch_size = 8  # 한 번에 8개씩 처리 (GPU 메모리에 맞게 조절)
test["generated_countermeasure"] = (
    test.groupby(test.index // batch_size, group_keys=False)
    .apply(generate_countermeasure_batch)
    .explode()
    .reset_index(drop=True)
)

In [ ]:
test.head()

In [ ]:
import joblib

joblib.dump(test["generated_countermeasure"].values, 'generated_countermeasure.pkl')

In [ ]:
embedding_model_name = "jhgan/ko-sbert-sts"
embedding = SentenceTransformer(embedding_model_name)

final_pred_embeddings = embedding.encode(test["generated_countermeasure"])

In [ ]:
submission.iloc[:,1] = test["generated_countermeasure"]
submission.iloc[:,2:] = final_pred_embeddings
submission.head()

In [ ]:
submission.to_csv('./final_submission.csv', index=False, encoding='utf-8-sig')

## LangChain RAG

In [ ]:
# from langchain.embeddings import HuggingFaceEmbeddings
# from langchain.vectorstores import FAISS
# from langchain.prompts import PromptTemplate
# from langchain.chains import RetrievalQA
# from langchain.llms import HuggingFacePipeline

# from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# import torch

In [ ]:
# combined_train_data = train.apply(
#     lambda row: {
#       'question': (
#           f"공사종류는 {row['공사종류(대분류)']}, {row['공사종류(중분류)']} "
#           f"공종은 {row['공종(대분류)']}, {row['공종(중분류)']} "
#           f"장소는 {row['장소(대분류)']}, {row['장소(중분류)']} "
#           f"사고객체는 {row['사고객체(대분류)']}, {row['사고객체(중분류)']} "
#           f"인적사고는 {row['인적사고']} "
#           f"물적사고는 {row['물적사고']} "
#           f"작업 프로세스는 {row['작업프로세스']} "
#           f"사고원인은 {row['사고원인']} "
#           f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
#       ),
#       'answer': row['재발방지대책 및 향후조치계획']
#     }, axis=1
# )

# combined_train_data = pd.DataFrame(list(combined_train_data))

# combined_test_data = test.apply(
#     lambda row: {
#         'question': (
#           f"공사종류는 {row['공사종류(대분류)']}, {row['공사종류(중분류)']} "
#           f"공종은 {row['공종(대분류)']}, {row['공종(중분류)']} "
#           f"장소는 {row['장소(대분류)']}, {row['장소(중분류)']} "
#           f"사고객체는 {row['사고객체(대분류)']}, {row['사고객체(중분류)']} "
#           f"인적사고는 {row['인적사고']} "
#           f"물적사고는 {row['물적사고']} "
#           f"작업 프로세스는 {row['작업프로세스']} "
#           f"사고원인은 {row['사고원인']} "
#           f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
#       ),
#     }, axis=1
# )

# combined_test_data = pd.DataFrame(list(combined_test_data))

In [ ]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# model_name = "NCSOFT/Llama-VARCO-8B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

In [ ]:
# combined_train_data

In [ ]:
# train_documents = [
#     f"질문: {row['question']}\n답변: {row['answer']}"
#     for _, row in combined_train_data.iterrows()
# ]

In [ ]:
# embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

# embeddings = HuggingFaceEmbeddings(
#     model_name="jhgan/ko-sbert-nli",
#     # model_kwargs={'device': 'cpu'},
#     # encode_kwargs={'normalize_embeddings': True}
# )

# # 의미 기반 검색
# vector_store = FAISS.from_texts(train_documents, embeddings)
# vector_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

# # BM25 키워드 기반 검색
# bm25_retriever = BM25Retriever.from_texts(train_documents)
# bm25_retriever.k = 5

# # Hybrid 검색 (FAISS 50% + BM25 50%)
# retriever = EnsembleRetriever(
#     retrievers=[vector_store.as_retriever(), bm25_retriever],
#     weights=[0.5, 0.5]
# )

# # text_generation_pipeline = pipeline(
# #     model=model,
# #     tokenizer=tokenizer,
# #     task="text-generation",
# #     do_sample=True,  # 샘플링 적용
# #     temperature=0.7, # 0.1
# #     # return_full_text=False,
# #     max_length=256, # 응답 길이 제한
# #     # max_new_tokens=64,
# # )

# text_generation_pipeline = pipeline(
#     model=model,
#     tokenizer=tokenizer,
#     task="text-generation",
#     do_sample=True,  # sampling 활성화
#     temperature=0.1,
#     return_full_text=False,
#     max_new_tokens=64,
# )

# prompt_template = """
# ### 지침: 당신은 건설 안전 전문가입니다.
# 질문에 대한 답변을 핵심 내용만 요약하여 간략하게 작성하세요.
# - 서론, 배경 설명 또는 추가 설명을 절대 포함하지 마세요.
# - 다음과 같은 조치를 취할 것을 제안합니다: 와 같은 내용을 포함하지 마세요.

# {context}

# ### 질문:
# {question}

# [/INST]

# """

# # 커스텀 프롬프트 생성
# prompt = PromptTemplate(
#     input_variables=["context", "question"],
#     template=prompt_template,
# )

# llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# hybrid_retriever = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5])

# qa_chain = RetrievalQA.from_chain_type(
#     llm=HuggingFacePipeline(pipeline=text_generation_pipeline),
#     chain_type="stuff",  # 간단한 context 조합 방식 사용 # map_rerank
#     retriever=hybrid_retriever,  # Hybrid Search 적용
#     return_source_documents=True,
#     chain_type_kwargs={"prompt": prompt}  # 커스텀 프롬프트 사용
# )

In [ ]:
# # 테스트 실행 및 결과 저장
# test_results = []

# print("테스트 실행 시작... 총 테스트 샘플 수:", len(combined_test_data))

# for idx, row in combined_test_data.iterrows():
#     # 50개당 한 번 진행 상황 출력
#     if (idx + 1) % 50 == 0 or idx == 0:
#         print(f"\n[샘플 {idx + 1}/{len(combined_test_data)}] 진행 중...")

#     # RAG 체인 호출 및 결과 생성
#     prevention_result = qa_chain.invoke(row['question'])

#     # 결과 저장
#     result_text = prevention_result['result']
#     test_results.append(result_text)

# print("\n테스트 실행 완료! 총 결과 수:", len(test_results))

In [ ]:
# # 테스트 데이터 검색 예시
# query = "특정 공사종류에서 발생할 수 있는 사고와 대책은?"
# result = qa_chain({"query": query})
# print(result["result"])

embedding_model_name = "jhgan/ko-sbert-sts" 임베딩 모델 꼭 이것으로 사용

## LLM

In [ ]:
# model_name = 'meta-llama/Llama-2-7b-hf'
# model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4it=True, device_map='auto')

# lora_config = LoraConfig(
#     r=16, lora_alpha=32, target_modules=['q_prog', 'v_proj'], lora_droupout=0.05
# )

# model = get_peft_model(model, lora_config)